In [1]:
from datasets import load_dataset

DocRed = load_dataset("thunlp/docred", trust_remote_code=True)

In [2]:
DocRed

DatasetDict({
    validation: Dataset({
        features: ['title', 'sents', 'vertexSet', 'labels'],
        num_rows: 998
    })
    test: Dataset({
        features: ['title', 'sents', 'vertexSet', 'labels'],
        num_rows: 1000
    })
    train_annotated: Dataset({
        features: ['title', 'sents', 'vertexSet', 'labels'],
        num_rows: 3053
    })
    train_distant: Dataset({
        features: ['title', 'sents', 'vertexSet', 'labels'],
        num_rows: 101873
    })
})

In [3]:
ner_tags = set()
for sample in DocRed['train_annotated']:
    for sent_entities in sample['vertexSet']:
        for entity in sent_entities:
            ner_tags.add(entity['type'])

print(ner_tags)

{'NUM', 'MISC', 'PER', 'ORG', 'TIME', 'LOC'}


In [4]:
label2id = {
    'O': 0, 
    'B-PER': 1, 
    'I-PER': 2, 
    'B-ORG': 3, 
    'I-ORG': 4, 
    'B-LOC': 5, 
    'I-LOC': 6, 
    'B-MISC': 7, 
    'I-MISC': 8,
    'B-NUM': 9,
    'I-NUM': 10,
    'B-TIME': 11,
    'I-TIME': 12
}
id2label = {id: label for label, id in label2id.items()}
id2label

{0: 'O',
 1: 'B-PER',
 2: 'I-PER',
 3: 'B-ORG',
 4: 'I-ORG',
 5: 'B-LOC',
 6: 'I-LOC',
 7: 'B-MISC',
 8: 'I-MISC',
 9: 'B-NUM',
 10: 'I-NUM',
 11: 'B-TIME',
 12: 'I-TIME'}

In [5]:
DocRed['train_annotated'][0]

{'title': 'AirAsia Zest',
 'sents': [['Zest',
   'Airways',
   ',',
   'Inc.',
   'operated',
   'as',
   'AirAsia',
   'Zest',
   '(',
   'formerly',
   'Asian',
   'Spirit',
   'and',
   'Zest',
   'Air',
   ')',
   ',',
   'was',
   'a',
   'low',
   '-',
   'cost',
   'airline',
   'based',
   'at',
   'the',
   'Ninoy',
   'Aquino',
   'International',
   'Airport',
   'in',
   'Pasay',
   'City',
   ',',
   'Metro',
   'Manila',
   'in',
   'the',
   'Philippines',
   '.'],
  ['It',
   'operated',
   'scheduled',
   'domestic',
   'and',
   'international',
   'tourist',
   'services',
   ',',
   'mainly',
   'feeder',
   'services',
   'linking',
   'Manila',
   'and',
   'Cebu',
   'with',
   '24',
   'domestic',
   'destinations',
   'in',
   'support',
   'of',
   'the',
   'trunk',
   'route',
   'operations',
   'of',
   'other',
   'airlines',
   '.'],
  ['In',
   '2013',
   ',',
   'the',
   'airline',
   'became',
   'an',
   'affiliate',
   'of',
   'Philippines',
   'A

In [6]:
def docred_to_bio_tags(example) -> list:
    """
    Convert DocRed vertexSet to BIO tags for all tokens in the document.

    Args:
        example: A single example from the DocRed dataset.

    Returns:
        list of BIO tags mathching the flattened token list
    """
    # Flatten all tokens in the document
    all_tokens = [tok for sent in example['sents'] for tok in sent]
    bio_tags = ['O'] * len(all_tokens)

    # Calculate sentence offsets to map entity positions to flattened token list
    global_offset = 0
    sent_offsets = []
    for sent in example['sents']:
        sent_offsets.append(global_offset)
        global_offset += len(sent)

    # Iterate over each entity cluster
    for entity_cluster in example['vertexSet']:
        # All entities in the cluster share the same type
        entity_type = entity_cluster[0]['type']

        # Assign BIO tags for each entity in the cluster
        for entity in entity_cluster:
            sent_id = entity['sent_id']
            start_pos, end_pos = entity['pos']

            for i in range(start_pos, end_pos):
                global_i = sent_offsets[sent_id] + i
                prefix = 'B-' if i == start_pos else 'I-'
                bio_tags[global_i] = f"{prefix}{entity_type}"

    return bio_tags


In [7]:
my_example = {
    'title': 'Test',
    'sents': [['Radek', 'was', 'born', 'in', 'Prague', '.'],
              ['Radek', 'is', '24', 'years', 'old', '.'],
              ['Prague', 'is', 'the', 'capital', 'of', 'Czechia', '.']],
    'vertexSet': [
        [  # Entity cluster 1: Radek (PER)
            {'name': 'Radek', 'sent_id': 0, 'pos': [0, 1], 'type': 'PER'},
            {'name': 'Radek', 'sent_id': 1, 'pos': [0, 1], 'type': 'PER'}
        ],
        [  # Entity cluster 2: Prague (LOC)
            {'name': 'Prague', 'sent_id': 0, 'pos': [4, 5], 'type': 'LOC'},
            {'name': 'Prague', 'sent_id': 2, 'pos': [0, 1], 'type': 'LOC'}
        ],
        [  # Entity cluster 3: 24 years (NUM)
            {'name': '24 years', 'sent_id': 1, 'pos': [2, 4], 'type': 'NUM'}
        ],
        [  # Entity cluster 4: Czechia (LOC)
            {'name': 'Czechia', 'sent_id': 2, 'pos': [5, 6], 'type': 'LOC'}
        ]
    ]
}

In [8]:
print(' '.join([tok for sent in my_example['sents'] for tok in sent]))
print(docred_to_bio_tags(my_example))

Radek was born in Prague . Radek is 24 years old . Prague is the capital of Czechia .
['B-PER', 'O', 'O', 'O', 'B-LOC', 'O', 'B-PER', 'O', 'B-NUM', 'I-NUM', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'B-LOC', 'O']


In [9]:
from openai import OpenAI

client = OpenAI(
    base_url = "http://n30:9069/v1",
    # base_url = "http://h02:9069/v1",
    # base_url = "http://n23:9069/v1",
    api_key = "ollama"
)

models = client.models.list()
models = sorted(models.data, key=lambda m: m.id)
for model in models:
    print(model.id)

gemma3:1b
gemma3:27b
gemma3:4b
gpt-oss:120b
gpt-oss:20b
llama3.1:70b
llama3.1:8b
qwen3:32b
qwen3:8b


In [ ]:
import sys, os
import evaluate
from tqdm import tqdm
import pandas as pd
from utils.system_prompts import *
from datasets import load_dataset
import time
from utils.context_matching_utils import json_safe_parse, assign_entities_from_context

# Load seqeval for evaluation
seqeval = evaluate.load("seqeval")

# Experiment parameters
MAX_EXAMPLES = 50 # Full evaluation
N_ITERS = 1
EVAL_INTERVAL = 10 # Log after every 10 iterations
BATCH_SIZE = 1
FUZZY = True
FUZZY_THRESHOLD = 0.6

# Define system prompts to evaluate
SYSTEM_PROMPT_DOCRED = """
You are an expert at named entity recognition. Given an input text, extract all named entities along with their types and surrounding context.
Do not extract nested entities, only the outermost ones.

IMPORTANT: If an entity appears multiple times, but with different surrounding context, extract each occurrence separately.

The possible labels for named entities are:
PER: names of people (different languages, nicknames, usernames, fictional characters, titles with names, etc.)
LOC: names of cities, countries, landmarks, geographical features, addresses, etc.
ORG: names of companies, institutions, agencies, teams, etc.
NUM: numerical expressions (counts, quantities, ages, etc.) with units (if applicable).
TIME: temporal expressions (dates, times, durations, years, etc.).
MISC: everythin else that could be considered a named entity (languages, nationalities, etc.).

The order of labeling is PER, LOC, ORG, NUM, TIME, MISC.

Output format:
Return a JSON array of objects:
[
    { "entity": "ENTITY", "label": "ENTITY_LABEL", "context": "SURROUNDING_CONTEXT" },
    ...
]

Rules:
- "entity" must exactly match the original substring from the input text.
- "label" must be one of the specified entity labels.
- "context" must be a short snippet (4-8 words) from the input text that contains the entity and a few neighboring words.
- The entity must be included in the context snippet
- If the entity is at the beginning or end of the text, use only the available neighboring words.
- If there are no named entities, output an empty JSON array [].

Example:
Input text: "John Doe , a software engineer at OpenAI , moved to San Francisco on September 6 , 2020 . He is 30 years old ."
Output:
```json
[
    { "entity": "John Doe", "label": "PER", "context": "John Doe , a software engineer" },
    { "entity": "OpenAI", "label": "ORG", "context": "software engineer at OpenAI , moved" },
    { "entity": "San Francisco", "label": "LOC", "context": "moved to San Francisco on September" },
    { "entity": "September 6 , 2020", "label": "TIME", "context": "on September 6 , 2020" },
    { "entity": "30 years", "label": "NUM", "context": "He is 30 years old ." }
]
```

IMPORTANT: Only output the JSON array. DO NOT add any explanations. Follow the format exactly.
"""
prompts = {
    SYSTEM_PROMPT_DOCRED: "SYSTEM_PROMPT_DOCRED"
}

# Load DocRed dataset
dataset = load_dataset("thunlp/docred", trust_remote_code=True)
dataset = dataset['validation']

# if MAX_EXAMPLES:
#     dataset = dataset.select(range(MAX_EXAMPLES))

print(f"Total examples in dataset: {MAX_EXAMPLES if MAX_EXAMPLES else len(dataset)}")
print(f"Number of iterations: {N_ITERS}")
print(f"\nBATCH_SIZE: {BATCH_SIZE}")
print(f"FUZZY mode: {FUZZY}, FUZZY_THRESHOLD: {FUZZY_THRESHOLD}\n")

txt_path = f"/home/stulcrad/master_thesis/NER_results/DocRed/Txt/ner_document_context_{BATCH_SIZE}_BATCHSZ.txt" if not FUZZY else \
    f"/home/stulcrad/master_thesis/NER_results/DocRed/Txt/ner_document_context_fuzzy_{BATCH_SIZE}_BATCHSZ.txt"

csv_path = txt_path.replace("/Txt/", "/Csv/").replace(".txt", ".csv")

print(f"Text path: {txt_path}")
print(f"Csv path: {csv_path}")

all_results = []
# Main evaluation loop, iterating over prompts and models
for prompt in prompts.keys():
    print(f"\n===== Using system prompt: {prompts[prompt]} =====\n", flush=True)

    for model_name in ["qwen3:8b", "gpt-oss:20b", "llama3.1:8b"]:
        print(f"\n==== Evaluating model: {model_name} ====", flush=True)
        
        exp_metrics = []

        # Repeat experiments for statistical significance
        for exp_id in range(N_ITERS):
            print(f"\n--- Running experiment {exp_id+1}/{N_ITERS} ---\n")

            # Random sample of dataset (without replacement)
            sampled_dataset = dataset.shuffle(seed=42+exp_id).select(range(MAX_EXAMPLES))
        
            # Low reasoning effort for faster responses
            system_prompt = prompt
            if model_name.startswith("qwen"):
                system_prompt += "\n\\no_think"
                pass

            start_time = time.time()
            true_entities, pred_entities = [], []

            # Process dataset in batches
            num_batches = (len(sampled_dataset) + BATCH_SIZE - 1) // BATCH_SIZE
            for batch_idx in tqdm(range(num_batches), file=sys.stdout):

                # batch = dataset[batch_idx * BATCH_SIZE : (batch_idx + 1) * BATCH_SIZE]
                batch = sampled_dataset.select(range(batch_idx * BATCH_SIZE, min((batch_idx + 1) * BATCH_SIZE, len(sampled_dataset))))

                if len(batch) == 0:
                    continue

                # Prepare full text and gold tags for the batch
                batch_tokens = []
                batch_gold_tags = []
                for example in batch:
                    # 1) Flatten DocRed tokens
                    tokens = [tok for sent in example['sents'] for tok in sent]
                    batch_tokens.extend(tokens)

                    # 2) Get BIO tags
                    bio_tags = docred_to_bio_tags(example)
                    batch_gold_tags.extend(bio_tags)
                
                # Join tokens to form full text input
                full_text = " ".join(batch_tokens)

                # print(f"\nFull text: {full_text}")
                # for token, tag in zip(batch_tokens, batch_gold_tags):
                #     print(f"{token}\t{tag}")

                # Prepare and send request to the model
                try:
                    req_kwargs = dict(model=model_name)
                    req_kwargs['messages'] = [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": full_text}
                    ]
                    req_kwargs['temperature'] = 0.3
                    if model_name.startswith("gpt-oss"):
                        req_kwargs['reasoning_effort'] = "low"

                    response = client.chat.completions.create(**req_kwargs)
                    content = response.choices[0].message.content.strip()
                    pred_json = json_safe_parse(content)
                    # print(f"\nRaw model output:\n{content}")
                    print(f"\nParsed JSON output:\n{pred_json}")
                except Exception as e:
                    print(f"Error processing batch: {e}")
                    pred_json = []

                # Assign BIO tags based on predicted entities and contexts
                pred_tags = assign_entities_from_context(batch_tokens, pred_json, fuzzy=FUZZY, fuzzy_threshold=FUZZY_THRESHOLD)

                # print(f"\nPredicted tags:")
                # for token, tag in zip(batch_tokens, pred_tags):
                #     print(f"{token}\t{tag}")
                # time.sleep(5)

                # Collect true and predicted entities for evaluation
                true_entities.append(batch_gold_tags)
                pred_entities.append(pred_tags)

                # Periodic logging
                if (batch_idx + 1) % EVAL_INTERVAL == 0:
                    metrics_partial = seqeval.compute(predictions=pred_entities, references=true_entities,
                                                    scheme="IOB2", mode="strict", zero_division=0)
                    elapsed = time.time() - start_time
                    tqdm.write(
                        f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] "
                        f"{model_name} progress: {batch_idx + 1}/{num_batches} "
                        f"({(batch_idx+1)/num_batches*100:.2f}%) | "
                        f"F1={metrics_partial['overall_f1']:.4f}, "
                        f"P={metrics_partial['overall_precision']:.4f}, "
                        f"R={metrics_partial['overall_recall']:.4f} | "
                        f"Acc={metrics_partial['overall_accuracy']:.4f} | "
                        f"Elapsed: {elapsed/60:.1f} min",
                        file=sys.stdout,
                    )

            # Compute metrics for this experiment
            metrics = seqeval.compute(predictions=pred_entities, references=true_entities,
                                    scheme="IOB2", mode="strict", zero_division=0)
            exp_duration = (time.time() - start_time) / 60.0
            exp_metrics.append({
                "precision": metrics["overall_precision"],
                "recall": metrics["overall_recall"],
                "f1": metrics["overall_f1"],
                "accuracy": metrics["overall_accuracy"],
                "elapsed_minute": exp_duration
            })

        # Average metrics over experiments
        avg_precision = sum(m["precision"] for m in exp_metrics) / N_ITERS
        avg_recall = sum(m["recall"] for m in exp_metrics) / N_ITERS
        avg_f1 = sum(m["f1"] for m in exp_metrics) / N_ITERS
        avg_accuracy = sum(m["accuracy"] for m in exp_metrics) / N_ITERS
        avg_elapsed = sum(m["elapsed_minute"] for m in exp_metrics) / N_ITERS
        
        all_results.append({
            "system_prompt": prompts[prompt],
            "model": model_name,
            "precision": round(avg_precision, 5),
            "recall": round(avg_recall, 5),
            "f1": round(avg_f1, 5),
            "accuracy": round(avg_accuracy, 5),
            "elapsed_minute": round(avg_elapsed, 3)
        })


df = pd.DataFrame(all_results)

import os
os.makedirs(os.path.dirname(txt_path), exist_ok=True)
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

with open(txt_path, "w") as f:
    f.write(df.to_string(index=False))

df.to_csv(csv_path, index=False, encoding="utf-8")

print(f"Results saved to {txt_path} and {csv_path}") 


Total examples in dataset: 50
Number of iterations: 1

BATCH_SIZE: 1
FUZZY mode: True, FUZZY_THRESHOLD: 0.6

Text path: /home/stulcrad/master_thesis/NER_results/DocRed/Txt/ner_document_context_fuzzy_1_BATCHSZ.txt
Csv path: /home/stulcrad/master_thesis/NER_results/DocRed/Csv/ner_document_context_fuzzy_1_BATCHSZ.csv

===== Using system prompt: SYSTEM_PROMPT_DOCRED =====


==== Evaluating model: qwen3:8b ====

--- Running experiment 1/1 ---

  0%|          | 0/50 [00:00<?, ?it/s]
Parsed JSON output:
[{'entity': 'Gavin DeGraw', 'label': 'PER', 'context': 'American recording artist Gavin DeGraw'}, {'entity': 'Sweeter', 'label': 'ORG', 'context': 'fourth studio album , Sweeter'}, {'entity': 'September 6 , 2011', 'label': 'TIME', 'context': 'released into the iTunes Store on September 6 , 2011'}, {'entity': 'September 24 , 2012', 'label': 'TIME', 'context': 'final single from the album in the United States , on September 24 , 2012'}, {'entity': 'One Tree Hill', 'label': 'LOC', 'context': 'las